In [ ]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Verifico scheda video e versione compilatore NVCC disponibile nell'ambiente**
- in questo caso A100 con 80Gb di VRAM

In [2]:
!nvidia-smi
!nvcc --version

Sun Sep 13 16:08:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os

# scarico la libreria trasformers utilizzata nella Repo LLaVA-NeXT
#(4.37.2 è l'ultima compatibile con apply_chunking_to_forward e LlavaLlamaForCausalLM)
!pip install "transformers==4.37.2" --quiet

# clono il repo ufficiale lmms-lab/LLaVA-NeXT
if not os.path.exists("/content/LLaVA-NeXT"):
    !git clone https://github.com/LLaVA-VL/LLaVA-NeXT.git /content/LLaVA-NeXT

%cd /content/LLaVA-NeXT

# installo dipendenze del repo (senza sovrascrivere la versione di transformers pinnata)
!pip install -e ".[train]" --no-deps --quiet
!pip install av accelerate peft timm einops sentencepiece --quiet

print("Setup completato — transformers:", end=" ")
import transformers; print(transformers.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 140.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 145.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.
Cloning into '/content/LLaVA-NeXT'...
remote: Enumerating objects: 6643, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 6643 (delta 22), reused 12 (delta 12), pack-reused 6609 (from 3)
Receiving ob

In [4]:
import sys
import torch

# aggiungo il repo su Google Drive
if "/content/LLaVA-NeXT" not in sys.path:
    sys.path.insert(0, "/content/LLaVA-NeXT")

# Loader ufficiale lmms-lab — carica pesi, tokenizer e image_processor
# con le chiavi corrette per LLaVA-Video-7B-Qwen2
from llava.model.builder import load_pretrained_model

model_path = "lmms-lab/LLaVA-Video-7B-Qwen2"
model_name = "llava_qwen"   # nome architettura richiesto dal builder


# carico il modello
tokenizer, model, image_processor, max_length = load_pretrained_model(
    model_path,
    None,           # model_base: None per modelli stand-alone
    model_name,
    torch_dtype="bfloat16",
    device_map="auto",
    attn_implementation="sdpa",
)


# mettiamo in modalità evaluation
model.eval()
device = next(model.parameters()).device
print(f"LLaVA-Video-7B-Qwen2 caricato su {device}")


/content/LLaVA-NeXT/llava/model/llava_arch.py:215: SyntaxWarning: "is not" with 'int' literal. Did you mean "!="?
  if slower_img_feat is not 0:


Please install pyav to use video processing functions.
OpenCLIP not installed
Loaded LLaVA model: lmms-lab/LLaVA-Video-7B-Qwen2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings.patch_embedding.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

Model Class: LlavaQwenForCausalLM
LLaVA-Video-7B-Qwen2 caricato su cuda:0


---
# Parte 1 — Riproduzione della baseline HD-EPIC (LLaVA-Video 7B)

Stesso identico protocollo del notebook `01_videollama2_inference.ipynb`, applicato al
secondo modello. Il valore da verificare e' quello del paper HD-EPIC  per LLaVA-Video 7B: **38.7%**, contro un random baseline del 20%.


L'unica parte specifica di questo notebook e' la funzione `llava_infer`.

## 1.1 Motore di valutazione

La cella qui sotto scrive il modulo (per inferenza e creazione report) e lo importa. 

In [5]:
%%writefile /content/hdepic_eval.py
"""
hdepic_eval - motore di valutazione per HD-EPIC VQA, categoria Nutrition.

Usato da entrambi i notebook di inferenza (VideoLLaMA2 e LLaVA-Video): tenere
prompt, parsing e metriche in un unico posto e' cio' che rende confrontabili i
risultati dei due modelli.

La parte specifica del modello e' tutta in `infer_fn(visual, prompt) -> str`.

Protocollo (fedele al paper HD-EPIC):
  - domande a 5 scelte, risposta come LETTERA (A-E)
  - accuracy di categoria = MEDIA DELLE ACCURACY PER PROTOTIPO
  - le domande con file mancanti o corrotti sono ESCLUSE, non contate sbagliate
"""
import json, os, re, glob, unicodedata
import pandas as pd

VERSION = "1.0"
LETTERS = "ABCDEFGH"

# Valori di riferimento del paper HD-EPIC (Tab. 2, colonna Nutrition, % accuracy)
PAPER_NUTRITION = {
    "VideoLLaMA 2 7B":       32.7,
    "LLaVA-Video 7B":        38.7,
    "Llama 3.2 90B (blind)": 36.7,
    "Gemini Pro":            34.7,
    "Human":                 85.0,
    "Random":                20.0,
}

# task -> (file di annotation, modalita' di input nativa)
TASK_FILES = {
    "video_nutrition_estimation": ("nutrition_video_nutrition_estimation.json", "video"),
    "nutrition_change":           ("nutrition_nutrition_change.json",           "video"),
    "image_nutrition_estimation": ("nutrition_image_nutrition_estimation.json", "images"),
}
VIDEO_TASKS = [t for t, (_, m) in TASK_FILES.items() if m == "video"]


# ============================================================ indici dei file
def build_video_index(roots):
    """Indicizza TUTTI gli mp4 sotto `roots` -> {'sampled': {id: path}, 'full': {id: path}}.

    Indicizzare globalmente invece che cartella-per-task e' importante: lo stesso
    video puo' servire a piu' task ed essere stato scaricato in una sola cartella.
    """
    idx = {"sampled": {}, "full": {}}
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True):
            b = os.path.basename(p)
            if b.endswith("_sampled.mp4"):
                idx["sampled"].setdefault(b[:-len("_sampled.mp4")], p)
            else:
                idx["full"].setdefault(b[:-len(".mp4")], p)
    return idx


def build_panels_index(roots):
    """Indicizza le cartelle di pannelli -> {video_id: [path dei panel ordinati]}.

    Struttura attesa (output di mosaic.py): <root>/<video_id>/<video_id>_panel_0000.jpg
    """
    idx = {}
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d in sorted(os.listdir(root)):
            full = os.path.join(root, d)
            if not os.path.isdir(full):
                continue
            panels = sorted(glob.glob(os.path.join(full, "*.jpg")) +
                            glob.glob(os.path.join(full, "*.jpeg")) +
                            glob.glob(os.path.join(full, "*.png")))
            if panels:
                idx.setdefault(d, panels)
    return idx


def resolve_video(video_id, index, prefer="sampled"):
    """-> (path, sorgente) con sorgente in {'sampled','full',None}."""
    order = ("sampled", "full") if prefer == "sampled" else ("full", "sampled")
    for src in order:
        if video_id in index[src]:
            return index[src][video_id], src
    return None, None


def hhmmss_to_sec(t):
    """'00:13:43.393' -> 823.393"""
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def load_annotations(ann_dir, task):
    with open(os.path.join(ann_dir, TASK_FILES[task][0])) as f:
        return json.load(f)


# =================================================================== prompt
ANSWER_INSTRUCTION = "Answer with the option's letter from the given choices directly."

# Prompt di dominio opzionale: NON usarlo per il confronto col paper, che non ne ha uno.
SYSTEM_PROMPT = (
    "You are a precise video analysis assistant. Track ingredients, actions and "
    "weight measurements on digital scales to answer nutrition-related questions. "
    "Base your answer ONLY on the visual evidence provided."
)


def build_prompt(question, choices, system=None):
    """Prompt a lettere A-E. `system=None` = versione minimale, fedele al paper."""
    opts = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(choices))
    head = f"{system}\n\n" if system else ""
    return f"{head}{question}\n{opts}\n{ANSWER_INSTRUCTION}"


NUTRIENT_RE = re.compile(r"\b(?:highest|higher)\s+(calories|carbs|fat|protein)\b", re.I)


def q_type_of(task, question):
    """Tipo di domanda: il nutriente chiesto, oppure il nome del prototipo."""
    m = NUTRIENT_RE.search(question)
    if m:
        return m.group(1).lower()
    return "nutrition_change" if task == "nutrition_change" else "unknown"


# ================================================================== parsing
def _norm(s):
    s = unicodedata.normalize("NFKD", s).lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def parse_answer(response, choices):
    """Estrae l'indice della scelta -> (idx, metodo).

    A livelli, dal piu' stretto al piu' permissivo. Il metodo viene salvato nei
    risultati, cosi' e' sempre ispezionabile COME si e' arrivati a quell'indice.
    idx = -1 significa 'risposta non interpretabile'.
    """
    n = len(choices)
    valid = LETTERS[:n]
    s = (response or "").strip()
    if not s:
        return -1, "empty"

    # 1) inizia con la lettera: "C", "C.", "(C)", "C) spring greens"
    #    il lookahead evita di leggere la C di "Calories changed by..."
    m = re.match(r"^\W*\(?([A-Za-z])\)?(?=[\s.,:;)\-]|$)", s)
    if m and m.group(1).upper() in valid:
        return valid.index(m.group(1).upper()), "leading_letter"

    # 2) formula esplicita: "the answer is C", "Option B"
    m = re.search(r"\b(?:answer|option|choice)\s*(?:is|:)?\s*\(?([A-Za-z])\)?\b", s, re.I)
    if m and m.group(1).upper() in valid:
        return valid.index(m.group(1).upper()), "answer_is_letter"

    # 3) ha ricopiato il testo di una scelta invece della lettera
    ns = _norm(s)
    hits = [i for i, c in enumerate(choices) if _norm(c) and _norm(c) in ns]
    if len(hits) == 1:
        return hits[0], "choice_text"
    hits = [i for i, c in enumerate(choices) if ns and ns in _norm(c)]
    if len(hits) == 1:
        return hits[0], "choice_text_partial"

    # 4) una sola lettera valida isolata in tutto il testo
    cand = {x.upper() for x in re.findall(r"\b([A-Za-z])\b", s)} & set(valid)
    if len(cand) == 1:
        return valid.index(cand.pop()), "single_letter_in_text"

    # 5) ha risposto con una cifra nonostante il formato a lettere
    m = re.match(r"^\W*(\d)\b", s)
    if m and 1 <= int(m.group(1)) <= n:
        return int(m.group(1)) - 1, "digit_1indexed"

    return -1, "unparsed"


# ================================================= integrita' dati / pre-volo
def is_readable(path, cache=None):
    """Il file esiste ED e' decodificabile?

    Nel dataset scaricato ci sono mp4 da 0 byte e mp4 troncati (download
    interrotti): esistono ma non si aprono. Contarli come risposte sbagliate
    falserebbe l'accuracy, quindi vanno individuati PRIMA di girare.
    """
    if cache is not None and path in cache:
        return cache[path]
    ok = False
    if path and os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            import cv2
            cap = cv2.VideoCapture(path)
            ok = bool(cap.read()[0])
            cap.release()
        except Exception:
            ok = False
    if cache is not None:
        cache[path] = ok
    return ok


def _required_inputs(item, task, index, panels_index, input_mode, prefer):
    """-> [(video_id, risorsa)] necessaria a rispondere alla domanda.

    `risorsa` e' un path mp4, una lista di path di pannelli, o None se assente.
    """
    modality = TASK_FILES[task][1]
    out = []
    if modality == "images":
        # 5 fotogrammi a timestamp precisi: serve per forza il video integrale
        for key in sorted(item["inputs"], key=lambda k: int(re.search(r"\d+", k).group())):
            inp = item["inputs"][key]
            path, _ = resolve_video(inp["id"], index, prefer="full")
            out.append((inp["id"], path))
    elif input_mode == "panels":
        for inp in item["inputs"].values():
            out.append((inp["id"], (panels_index or {}).get(inp["id"])))
    else:
        for inp in item["inputs"].values():
            path, _ = resolve_video(inp["id"], index, prefer=prefer)
            out.append((inp["id"], path))
    return out


def usable_items(ann_dir, task, index, panels_index=None, input_mode="video",
                 prefer="sampled", cache=None):
    """Divide le domande in (utilizzabili, scartate-con-motivo)."""
    ann = load_annotations(ann_dir, task)
    ok, skipped = {}, {}
    for key, item in ann.items():
        bad = []
        for vid, res in _required_inputs(item, task, index, panels_index,
                                         input_mode, prefer):
            if isinstance(res, list):          # pannelli: ne basta almeno uno
                if not res:
                    bad.append(vid)
            elif not is_readable(res, cache):  # video: deve aprirsi davvero
                bad.append(vid)
        if bad:
            skipped[key] = sorted(set(bad))
        else:
            ok[key] = item
    return ok, skipped


def preflight(ann_dir, index, tasks=None, panels_index=None, input_mode="video",
              prefer="sampled", cache=None):
    """Tabella di copertura reale. Da stampare SEMPRE prima di un run."""
    cache = {} if cache is None else cache
    rows, details = [], {}
    for task in (tasks or list(TASK_FILES)):
        ann = load_annotations(ann_dir, task)
        ok, skipped = usable_items(ann_dir, task, index, panels_index,
                                   input_mode, prefer, cache)
        details[task] = skipped
        rows.append({"task": task,
                     "input": "frames" if TASK_FILES[task][1] == "images" else input_mode,
                     "domande": len(ann),
                     "utilizzabili": len(ok),
                     "scartate": len(skipped),
                     "file_inutilizzabili": len({v for ids in skipped.values() for v in ids})})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {"task": "TOTALE", "input": "", "domande": df["domande"].sum(),
                       "utilizzabili": df["utilizzabili"].sum(),
                       "scartate": df["scartate"].sum(), "file_inutilizzabili": ""}
    return df, details


# ======================================================== estrazione frame
def extract_frame(video_path, sec, out_path):
    """Salva il frame all'istante `sec`. Prova decord, poi ripiega su OpenCV.

    Serve per image_nutrition_estimation, dove ogni domanda mostra 5 fotogrammi
    presi da 5 video diversi a timestamp indicati nelle annotation.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    try:
        from decord import VideoReader
        from PIL import Image
        vr = VideoReader(video_path, num_threads=1)
        i = min(int(round(sec * vr.get_avg_fps())), len(vr) - 1)
        Image.fromarray(vr[i].asnumpy()).save(out_path, quality=95)
        return out_path
    except Exception:
        pass
    import cv2
    cap = cv2.VideoCapture(video_path)
    try:
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        i = int(round(sec * fps))
        if total:
            i = min(i, total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        if not ok:                                  # seek per indice fallito
            cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000.0)
            ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"frame non leggibile a {sec:.3f}s in {video_path}")
        cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    finally:
        cap.release()
    return out_path


def _visual_input(item, task, index, panels_index, input_mode, prefer,
                  frame_extractor, frame_cache):
    """Prepara cio' che va passato al modello -> (input, etichetta sorgente)."""
    if TASK_FILES[task][1] == "images":
        paths = []
        for key in sorted(item["inputs"], key=lambda k: int(re.search(r"\d+", k).group())):
            inp = item["inputs"][key]
            video, _ = resolve_video(inp["id"], index, prefer="full")
            sec = hhmmss_to_sec(inp["time"])
            out = os.path.join(frame_cache, f"{inp['id']}_{int(round(sec*1000)):09d}.jpg")
            if not os.path.exists(out):
                (frame_extractor or extract_frame)(video, sec, out)
            paths.append(out)
        return paths, "frames"
    vid = item["inputs"]["video 1"]["id"]
    if input_mode == "panels":
        return panels_index[vid], "panels"
    path, src = resolve_video(vid, index, prefer=prefer)
    return path, src


# ================================================================ inferenza
def run_task(task, ann_dir, index, infer_fn, out_jsonl, panels_index=None,
             input_mode="video", prefer="sampled", system=None,
             frame_extractor=None, frame_cache=None, limit=None,
             verbose=True, readable_cache=None):
    """Inferenza su un task, con ripresa da dove si era interrotto.

    Ogni risposta viene scritta su `out_jsonl` subito: se Colab si disconnette,
    rilanciare la cella riprende senza rifare le domande gia' fatte.
    """
    ann, skipped = usable_items(ann_dir, task, index, panels_index,
                                input_mode, prefer, readable_cache)
    if frame_cache:
        os.makedirs(frame_cache, exist_ok=True)
    if verbose:
        print(f"  {len(ann)} domande utilizzabili"
              + (f", {len(skipped)} escluse (file mancanti o corrotti)" if skipped else ""))

    done = {}
    if os.path.exists(out_jsonl):
        with open(out_jsonl) as f:
            for line in f:
                r = json.loads(line)
                done[r["key"]] = r
        if verbose and done:
            print(f"  ripresa: {len(done)} risposte gia' presenti")

    results = []
    for n, (key, item) in enumerate(ann.items()):
        if limit and n >= limit:
            break
        if key in done:
            results.append(done[key])
            continue

        question, choices, correct = item["question"], item["choices"], item["correct_idx"]
        rec = {"key": key, "task": task, "input_mode": input_mode,
               "q_type": q_type_of(task, question),
               "video_id": list(item["inputs"].values())[0]["id"],
               "n_choices": len(choices), "correct_idx": correct}
        try:
            visual, src = _visual_input(item, task, index, panels_index, input_mode,
                                        prefer, frame_extractor, frame_cache)
            response = infer_fn(visual, build_prompt(question, choices, system=system))
            idx, method = parse_answer(response, choices)
            rec.update(source=src, response=response, predicted_idx=idx,
                       parse_method=method, error=None)
        except Exception as e:
            rec.update(source=None, response=None, predicted_idx=-1,
                       parse_method="error", error=f"{type(e).__name__}: {e}")

        rec["is_correct"] = bool(rec["predicted_idx"] == correct)
        with open(out_jsonl, "a") as f:
            f.write(json.dumps(rec) + "\n")
        results.append(rec)

        if verbose:
            mark = "ERR" if rec["error"] else ("OK " if rec["is_correct"] else "KO ")
            shown = (rec["response"] or rec["error"] or "")[:55]
            print(f"  [{n+1}/{len(ann)}] {mark} {key.split('_')[-1]:>3} | "
                  f"pred={rec['predicted_idx']} gt={correct} | "
                  f"{rec['parse_method']:<18} | {shown!r}")

    df = pd.DataFrame(results)
    df.attrs["skipped"] = skipped
    return df


def run_all(tasks, ann_dir, index, infer_fn, out_dir, tag, **kw):
    """Gira piu' task e concatena i risultati."""
    os.makedirs(out_dir, exist_ok=True)
    out, skipped = [], {}
    for task in tasks:
        print(f"\n{'='*70}\n  TASK: {task}   (run: {tag})\n{'='*70}")
        df = run_task(task, ann_dir, index, infer_fn,
                      os.path.join(out_dir, f"{tag}_{task}.jsonl"), **kw)
        skipped[task] = df.attrs.get("skipped", {})
        out.append(df)
    allr = pd.concat(out, ignore_index=True)
    allr.attrs["skipped"] = skipped
    return allr


# ================================================================= metriche
def accuracy_report(df, label="", paper_ref=None, ann_dir=None, show_bias=True):
    """Report allineato al protocollo del paper.

    HD-EPIC Tab. 2 e' la MEDIA DELLE ACCURACY PER PROTOTIPO, non l'accuracy
    aggregata sulle domande: con prototipi da 50/50/100 i due numeri non
    coincidono. Qui vengono stampati entrambi, cosi' il confronto e' esplicito.
    """
    out = {}
    per_task = df.groupby("task")["is_correct"].agg(corrette="sum", valutate="count")
    per_task["accuracy_%"] = (per_task["corrette"] / per_task["valutate"] * 100).round(1)
    if ann_dir:
        per_task["su_totale"] = [len(load_annotations(ann_dir, t)) for t in per_task.index]
    out["per_task"] = per_task
    out["nutrition_paper"] = round(per_task["accuracy_%"].mean(), 1)
    out["nutrition_pooled"] = round(df["is_correct"].mean() * 100, 1)

    print("=" * 70)
    print(f"  {label}   (n = {len(df)} domande valutate)")
    print("=" * 70)
    print(per_task.to_string())
    print(f"\n  Nutrition, metrica del paper (media per prototipo) : {out['nutrition_paper']}%")
    print(f"  Nutrition, accuracy aggregata sulle domande        : {out['nutrition_pooled']}%")
    print(f"  Baseline random                                    : 20.0%")
    if paper_ref in PAPER_NUTRITION:
        ref = PAPER_NUTRITION[paper_ref]
        print(f"  Paper, {paper_ref:<28}: {ref}%   "
              f"(delta {out['nutrition_paper'] - ref:+.1f})")

    inval = df[df["predicted_idx"] < 0]
    print(f"\n  Risposte non interpretate o in errore: {len(inval)}/{len(df)} "
          f"({len(inval)/max(len(df),1)*100:.1f}%)")
    print("  Come e' stata letta la risposta:")
    print("   ", df["parse_method"].value_counts().to_dict())
    out["parse_methods"] = df["parse_method"].value_counts()

    by_type = df.groupby("q_type")["is_correct"].agg(corrette="sum", valutate="count")
    by_type["accuracy_%"] = (by_type["corrette"] / by_type["valutate"] * 100).round(1)
    print("\n  Accuracy per tipo di domanda:")
    print(by_type.to_string())
    out["per_q_type"] = by_type

    if show_bias:
        print("\n  Bias posizionale (% risposte su ciascuna lettera, vs ground truth):")
        bias = {}
        for task, g in df.groupby("task"):
            n = int(g["n_choices"].max())
            pred = g[g["predicted_idx"] >= 0]["predicted_idx"].value_counts(normalize=True)
            gt = g["correct_idx"].value_counts(normalize=True)
            row = (pd.DataFrame({"pred": pred, "gt": gt})
                     .reindex(range(n)).fillna(0) * 100).round(0).astype(int)
            bias[task] = row["pred"]
            fmt = lambda c: "[" + " ".join(f"{v:3d}" for v in row[c]) + "]"
            print(f"    {task:<28} pred(A-E)={fmt('pred')}  gt(A-E)={fmt('gt')}")
        out["bias"] = pd.DataFrame(bias)
    return out


def compare_runs(runs, label="Confronto"):
    """Confronto fra piu' run: {nome: DataFrame} -> tabella accuracy per task."""
    rows = {}
    for name, df in runs.items():
        acc = (df.groupby("task")["is_correct"].mean() * 100).round(1)
        acc["NUTRITION (media prototipi)"] = round(acc.mean(), 1)
        rows[name] = acc
    tab = pd.DataFrame(rows)
    if len(runs) == 2:
        a, b = list(runs)
        tab["delta"] = (tab[b] - tab[a]).round(1)
    print("=" * 70)
    print(f"  {label}")
    print("=" * 70)
    print(tab.to_string())
    return tab

Writing /content/hdepic_eval.py


## 1.2 Percorsi e indicizzazione dei file

Indicizzo i file

In [7]:
import os, glob

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
ANN_DIR   = BASE
OUT_DIR   = os.path.join(BASE, "risultati")
FRAME_DIR = os.path.join(BASE, "frames_image_nutrition_estimation")

VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
PANEL_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" in os.path.basename(d).lower())

print("cartelle video :", [os.path.basename(d) for d in VIDEO_ROOTS])
print("cartelle panels:", [os.path.basename(d) for d in PANEL_ROOTS])

VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
PANEL_INDEX = H.build_panels_index(PANEL_ROOTS)
print(f"\nindicizzati: {len(VIDEO_INDEX['sampled'])} video campionati, "
      f"{len(VIDEO_INDEX['full'])} video integrali, {len(PANEL_INDEX)} cartelle di pannelli")

READABLE = {}
cov, dettagli = H.preflight(ANN_DIR, VIDEO_INDEX, cache=READABLE)
print()
print(cov.to_string(index=False))

cartelle video : ['Videos Nutrition Change Sampled', 'Videos Nutrition Estimation Sampled', 'videos_image_nutrition_estimation_sampled']
cartelle panels: ['videos_image_nutrition_estimation_panels', 'videos_nutrition_change_panels', 'videos_nutrition_estimation_panels']

indicizzati: 46 video campionati, 0 video integrali, 46 cartelle di pannelli

                      task  input  domande  utilizzabili  scartate file_inutilizzabili
video_nutrition_estimation  video       50            46         4                   1
          nutrition_change  video       50            50         0                   0
image_nutrition_estimation frames      100            88        12                   4
                    TOTALE             200           184        16                    


## 1.3 Il modello

LLaVA-Video vuole i fotogrammi gia' estratti, quindi la funzione di inferenza deve gestire
due casi: il path di un mp4 (da cui campionare i frame con PyAV) e una lista di path di
immagini (i 5 fotogrammi del task `image_nutrition_estimation`, oppure i pannelli della
Parte 2).

**Budget di fotogrammi.** `N_FRAMES = 32` vale per entrambi i casi. Il numero non e'
arbitrario: i pannelli generati da `mosaic.py` sono 32 per video, quindi fissando 32 anche
per il video liscio il confronto della Parte 2 avviene a **parita' di finestra di contesto**,
che e' esattamente la condizione in cui il paper dei Video Panels rivendica il guadagno.
Con 32 frame lisci il modello copre 32 fotogrammi; con 32 pannelli 2x2 ne copre 128.

In [ ]:
import av, torch
import numpy as np
from PIL import Image

from llava.conversation import conv_templates
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.mm_utils import tokenizer_image_token

N_FRAMES = 32
MAX_NEW_TOKENS = 32
USE_SYSTEM_PROMPT = None     # None = prompt minimale, fedele al paper

# estrae i fotogrammi in quanto LlavaVideo vuole i fotogrammi già estratti
def sample_video_frames(video_path, n=N_FRAMES):
    """Estrae n fotogrammi uniformi con PyAV (il codebase ufficiale di LLaVA-Video usa PyAV)."""
    container = av.open(video_path)
    stream = container.streams.video[0]
    total = stream.frames
    if total == 0:                                  # metadato assente: li conto
        total = sum(1 for _ in container.decode(video=0))
        container.close()
        container = av.open(video_path)
    voluti = set(np.linspace(0, total - 1, min(n, total), dtype=int).tolist())
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in voluti:
            frames.append(frame.to_ndarray(format="rgb24"))
        if len(frames) == len(voluti):
            break
    container.close()
    return frames


def _immagini(visual, n=N_FRAMES):
    """Normalizza l'input visivo in una lista di PIL.Image."""
    if isinstance(visual, list):                    # frame estratti o pannelli
        paths = visual
        if len(paths) > n:                          # sottocampiono a parita' di budget
            sel = np.linspace(0, len(paths) - 1, n, dtype=int)
            paths = [paths[i] for i in sel]
        return [Image.open(p).convert("RGB") for p in paths]
    return [Image.fromarray(f) for f in sample_video_frames(visual, n)]


# inferenza per llava video
def llava_infer(visual, prompt):
    """visual: path di un mp4, oppure lista di path di immagini."""
    immagini = _immagini(visual)
    video_tensor = image_processor.preprocess(immagini, return_tensors="pt")["pixel_values"]
    video_tensor = video_tensor.to(device=device, dtype=torch.bfloat16)

    # il token immagine vale anche per i video nel codebase ufficiale
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], f"{DEFAULT_IMAGE_TOKEN}\n{prompt}")
    conv.append_message(conv.roles[1], None)
    input_ids = tokenizer_image_token(conv.get_prompt(), tokenizer,
                                      IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(device)

    with torch.no_grad():
        out = model.generate(input_ids, images=[video_tensor], image_sizes=None,
                             modalities=["video"],
                             do_sample=False, temperature=None,   # greedy: deterministico
                             max_new_tokens=MAX_NEW_TOKENS, use_cache=True)

    testo = tokenizer.decode(out[0], skip_special_tokens=False)
    marker = "<|im_start|>assistant\n"
    if marker in testo:
        return testo.split(marker)[-1].replace("<|im_end|>", "").strip()
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


print(f"pronto - {N_FRAMES} fotogrammi per domanda, decoding greedy")

pronto - 32 fotogrammi per domanda, decoding greedy


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 1.5 Inferenza completa

In [ ]:
baseline = H.run_all(list(H.TASK_FILES), ANN_DIR, VIDEO_INDEX, llava_infer,
                     out_dir=OUT_DIR, tag="llavavideo_baseline",
                     system=USE_SYSTEM_PROMPT,
                     frame_extractor=H.extract_frame, frame_cache=FRAME_DIR,
                     readable_cache=READABLE)#inferenza sui video (frame estratti con le funzioni precedenti)

print(f"\nCompletate {len(baseline)} domande")


  TASK: video_nutrition_estimation   (run: llavavideo_baseline)
  46 domande utilizzabili, 4 escluse (file mancanti o corrotti)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `None` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:407: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  [1/46] KO    0 | pred=0 gt=3 | leading_letter     | 'A'
  [2/46] KO    1 | pred=2 gt=0 | leading_letter     | 'C'
  [3/46] OK    2 | pred=4 gt=4 | leading_letter     | 'E'
  [4/46] OK    3 | pred=4 gt=4 | leading_letter     | 'E'
  [5/46] OK    4 | pred=1 gt=1 | leading_letter     | 'B'
  [6/46] OK    5 | pred=4 gt=4 | leading_letter     | 'E'
  [7/46] OK    6 | pred=2 gt=2 | leading_letter     | 'C'
  [8/46] KO    7 | pred=1 gt=3 | leading_letter     | 'B'
  [9/46] OK    8 | pred=3 gt=3 | leading_letter     | 'D'
  [10/46] KO    9 | pred=1 gt=4 | leading_letter     | 'B'
  [11/46] OK   10 | pred=2 gt=2 | leading_letter     | 'C'
  [12/46] OK   11 | pred=4 gt=4 | leading_letter     | 'E'
  [13/46] OK   12 | pred=1 gt=1 | leading_letter     | 'B'
  [14/46] KO   13 | pred=2 gt=0 | leading_letter     | 'C'
  [15/46] OK   14 | pred=0 gt=0 | leading_letter     | 'A'
  [16/46] OK   15 | pred=4 gt=4 | leading_letter     | 'E'
  [17/46] OK   16 | pred=0 gt=0 | leading_letter     | 'A'
  [18/

In [11]:
rep_base = H.accuracy_report(baseline,
                             label="LLaVA-Video 7B - baseline (video)",
                             paper_ref="LLaVA-Video 7B",
                             ann_dir=ANN_DIR)

  LLaVA-Video 7B - baseline (video)   (n = 184 domande valutate)
                            corrette  valutate  accuracy_%  su_totale
task                                                                 
image_nutrition_estimation        23        88        26.1        100
nutrition_change                  11        50        22.0         50
video_nutrition_estimation        33        46        71.7         50

  Nutrition, metrica del paper (media per prototipo) : 39.9%
  Nutrition, accuracy aggregata sulle domande        : 36.4%
  Baseline random                                    : 20.0%
  Paper, LLaVA-Video 7B              : 38.7%   (delta +1.2)

  Risposte non interpretate o in errore: 0/184 (0.0%)
  Come e' stata letta la risposta:
    {'leading_letter': 184}

  Accuracy per tipo di domanda:
                  corrette  valutate  accuracy_%
q_type                                          
calories                14        37        37.8
carbs                   15        29       

---
# Parte 2 — Video Panels

Esperimento basato sul Paper dei Panels: 32 pannelli 2x2 al posto di 32 fotogrammi lisci, cioe'
128 fotogrammi coperti invece di 32 a parita' di finestra di contesto.

Anche qui solo i due prototipi basati su video: `image_nutrition_estimation` ha come input
5 fotogrammi isolati presi da video diversi, non c'e' una sequenza temporale da impacchettare.

In [12]:
cov_p, dett_p = H.preflight(ANN_DIR, VIDEO_INDEX, tasks=H.VIDEO_TASKS,
                            panels_index=PANEL_INDEX, input_mode="panels",
                            cache=READABLE)
print(cov_p.to_string(index=False))

panels_run = H.run_all(H.VIDEO_TASKS, ANN_DIR, VIDEO_INDEX, llava_infer,
                       out_dir=OUT_DIR, tag="llavavideo_panels",
                       panels_index=PANEL_INDEX, input_mode="panels",
                       system=USE_SYSTEM_PROMPT, readable_cache=READABLE)

rep_pan = H.accuracy_report(panels_run,
                            label="LLaVA-Video 7B - Video Panels 2x2",
                            ann_dir=ANN_DIR)

                      task  input  domande  utilizzabili  scartate file_inutilizzabili
video_nutrition_estimation panels       50            46         4                   1
          nutrition_change panels       50            50         0                   0
                    TOTALE             100            96         4                    

  TASK: video_nutrition_estimation   (run: llavavideo_panels)
  46 domande utilizzabili, 4 escluse (file mancanti o corrotti)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `None` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:407: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  [1/46] OK    0 | pred=3 gt=3 | leading_letter     | 'D'
  [2/46] KO    1 | pred=2 gt=0 | leading_letter     | 'C'
  [3/46] OK    2 | pred=4 gt=4 | leading_letter     | 'E'
  [4/46] OK    3 | pred=4 gt=4 | leading_letter     | 'E'
  [5/46] OK    4 | pred=1 gt=1 | leading_letter     | 'B'
  [6/46] OK    5 | pred=4 gt=4 | leading_letter     | 'E'
  [7/46] OK    6 | pred=2 gt=2 | leading_letter     | 'C'
  [8/46] OK    7 | pred=3 gt=3 | leading_letter     | 'D'
  [9/46] OK    8 | pred=3 gt=3 | leading_letter     | 'D'
  [10/46] KO    9 | pred=1 gt=4 | leading_letter     | 'B'
  [11/46] OK   10 | pred=2 gt=2 | leading_letter     | 'C'
  [12/46] OK   11 | pred=4 gt=4 | leading_letter     | 'E'
  [13/46] OK   12 | pred=1 gt=1 | leading_letter     | 'B'
  [14/46] KO   13 | pred=2 gt=0 | leading_letter     | 'C'
  [15/46] KO   14 | pred=4 gt=0 | leading_letter     | 'E'
  [16/46] OK   15 | pred=4 gt=4 | leading_letter     | 'E'
  [17/46] OK   16 | pred=0 gt=0 | leading_letter     | 'A'
  [18/

In [13]:
comuni = set(panels_run["key"]) & set(baseline["key"])
b = baseline[baseline["key"].isin(comuni)]
p = panels_run[panels_run["key"].isin(comuni)]
print(f"domande confrontate a parita' di input: {len(comuni)}\n")

tab = H.compare_runs({"baseline (video)": b, "panels 2x2": p},
                     "LLaVA-Video 7B: video liscio vs pannelli 2x2")

domande confrontate a parita' di input: 96

  LLaVA-Video 7B: video liscio vs pannelli 2x2
                             baseline (video)  panels 2x2  delta
task                                                            
nutrition_change                         22.0        16.0   -6.0
video_nutrition_estimation               71.7        76.1    4.4
NUTRITION (media prototipi)              46.8        46.0   -0.8


## 2.1 Salvataggio dei risultati

I CSV finiscono nella stessa cartella di quelli di VideoLLaMA 2, con il nome del modello nel
prefisso: da li' si costruisce la tabella finale con le due righe da confrontare con il paper.

In [14]:
import pandas as pd

tutto = pd.concat([baseline.assign(run="baseline"),
                   panels_run.assign(run="panels")], ignore_index=True)
csv = os.path.join(OUT_DIR, "llavavideo_risultati_completi.csv")
tutto.to_csv(csv, index=False)

riepilogo = pd.DataFrame({
    "run": ["baseline (video)", "panels 2x2"],
    "nutrition_media_prototipi_%": [rep_base["nutrition_paper"], rep_pan["nutrition_paper"]],
    "nutrition_aggregata_%":       [rep_base["nutrition_pooled"], rep_pan["nutrition_pooled"]],
    "domande_valutate":            [len(baseline), len(panels_run)],
})
riepilogo.to_csv(os.path.join(OUT_DIR, "llavavideo_riepilogo.csv"), index=False)

print(riepilogo.to_string(index=False))
print(f"\nsalvato: {csv}")
print(f"paper, LLaVA-Video 7B su Nutrition: {H.PAPER_NUTRITION['LLaVA-Video 7B']}%  | random: 20.0%")

             run  nutrition_media_prototipi_%  nutrition_aggregata_%  domande_valutate
baseline (video)                         39.9                   36.4               184
      panels 2x2                         46.0                   44.8                96

salvato: /content/drive/MyDrive/Progetto Deep Learning 1/risultati/llavavideo_risultati_completi.csv
paper, LLaVA-Video 7B su Nutrition: 38.7%  | random: 20.0%
